In [ ]:
# Enron Spam Classification: Before and After Outlier Removal
# Objective
# This project performs a complete spam-classification workflow on the Enron spam dataset.
# The analysis is divided into two experiments:
# 1. Build and evaluate a classification model **before outlier removal**.


In [ ]:
# 1. Import Required Libraries


In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

RANDOM_STATE = 42

In [ ]:
# 2. Load the Dataset



In [ ]:
DATA_PATH = "enron_spam.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

In [ ]:
# 3. Understand the Dataset


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

In [ ]:
print("Statistical summary:")
display(df.describe(include="all"))

In [ ]:
# 4. Identify the Text and Label Columns
# The code below tries to identify common names for the email-text and target columns automatically.


In [ ]:
possible_text_columns = [
    "text", "email", "body", "message", "content",
    "email_text", "mail", "raw_text"
]

possible_label_columns = [
    "label", "class", "target", "spam", "category", "type"
]

def find_column(columns, candidates):
    lower_map = {str(c).lower().strip(): c for c in columns}

    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]

    for col in columns:
        col_lower = str(col).lower()
        for candidate in candidates:
            if candidate in col_lower:
                return col

    return None

TEXT_COLUMN = find_column(df.columns, possible_text_columns)
LABEL_COLUMN = find_column(df.columns, possible_label_columns)

print("Detected text column:", TEXT_COLUMN)
print("Detected label column:", LABEL_COLUMN)

# If the automatic detection is incorrect, replace the two values below.
# Example:
# TEXT_COLUMN = "text"
# LABEL_COLUMN = "label"

if TEXT_COLUMN is None or LABEL_COLUMN is None:
    raise ValueError(
        "Could not identify the text or label column. "
        "Inspect df.columns and manually set TEXT_COLUMN and LABEL_COLUMN."
    )

In [ ]:
# 5. Missing Value Analysis


In [ ]:
missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df)) * 100
})

display(
    missing_values.sort_values(
        by="Missing Values",
        ascending=False
    )
)

In [ ]:
# 6. Duplicate Analysis


In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

df = df.drop_duplicates().copy()

print("Dataset shape after duplicate removal:", df.shape)

In [ ]:
# 7. Inspect the Target Variable
# The class distribution is checked before preprocessing the labels.


In [ ]:
print("Unique label values:")
print(df[LABEL_COLUMN].unique())

print("\nClass distribution:")
print(df[LABEL_COLUMN].value_counts(dropna=False))

In [ ]:
# 8. Visualize the Class Distribution
# This visualization shows the number of spam and legitimate emails in the dataset.


In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x=LABEL_COLUMN
)

plt.title("Distribution of Email Classes")
plt.xlabel("Email Class")
plt.ylabel("Number of Emails")

plt.tight_layout()
plt.show()

In [ ]:
# 9. Missing Value Removal
# Empty email messages are also removed.


In [ ]:
rows_before = len(df)

df = df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
).copy()

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.strip()
)

df = df[df[TEXT_COLUMN] != ""].copy()

rows_after = len(df)

print("Rows before missing-value removal:", rows_before)
print("Rows after missing-value removal:", rows_after)
print("Rows removed:", rows_before - rows_after)

In [ ]:
# 10. Convert Labels to Binary Values
# For the classification model, we use:
# `0` = Ham / legitimate email
# `1` = Spam
# The conversion below handles common `spam`/`ham` labels as well as common numeric or text alternatives.


In [ ]:
def convert_label(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    spam_labels = {
        "spam", "1", "true", "yes", "junk", "phishing"
    }

    ham_labels = {
        "ham", "0", "false", "no", "legitimate",
        "normal", "non-spam", "nonspam", "not spam"
    }

    if value in spam_labels:
        return 1

    if value in ham_labels:
        return 0

    return np.nan


df["target"] = df[LABEL_COLUMN].apply(convert_label)

print("Converted target distribution:")
print(df["target"].value_counts(dropna=False))

In [ ]:
unknown_labels = df.loc[
    df["target"].isna(),
    LABEL_COLUMN
].dropna().unique()

if len(unknown_labels) > 0:
    print("Labels that were not recognized:")
    print(unknown_labels)
else:
    print("All labels were successfully recognized.")

In [ ]:
# 11. Remove Rows With Unrecognized Labels
# Any row whose class cannot be converted to 0 or 1 is removed because it cannot be used as a supervised target.


In [ ]:
df = df.dropna(subset=["target"]).copy()
df["target"] = df["target"].astype(int)

print("Dataset shape after target cleaning:", df.shape)
print("\nFinal target distribution:")
print(df["target"].value_counts())

In [ ]:
# 12. Text Preprocessing


In [ ]:
def clean_email(text):
    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace URLs with a common token
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_text"] = df[TEXT_COLUMN].apply(clean_email)

display(
    df[[TEXT_COLUMN, "clean_text"]].head()
)

In [ ]:
# 13. Feature Engineering


In [ ]:
df["character_count"] = (
    df[TEXT_COLUMN].astype(str).str.len()
)

df["word_count"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.split()
    .str.len()
)

df["sentence_count"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.count(r"[.!?]")
)

df["url_count"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.count(r"https?://|www\.")
)

df["digit_count"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.count(r"\d")
)

df["uppercase_count"] = df[TEXT_COLUMN].apply(
    lambda text: sum(
        1 for character in str(text)
        if character.isupper()
    )
)

df["special_character_count"] = df[TEXT_COLUMN].apply(
    lambda text: sum(
        1 for character in str(text)
        if not character.isalnum() and not character.isspace()
    )
)

feature_columns = [
    "character_count",
    "word_count",
    "sentence_count",
    "url_count",
    "digit_count",
    "uppercase_count",
    "special_character_count"
]

display(df[feature_columns].head())

In [ ]:
# 14. Numerical Feature Summary


In [ ]:
display(
    df[feature_columns]
    .describe()
    .T
)

In [ ]:
# 15. Exploratory Data Analysis
# The following visualizations examine differences between spam and legitimate emails.


In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="character_count",
    hue="target",
    bins=50,
    kde=True
)

plt.title("Distribution of Email Character Count")
plt.xlabel("Number of Characters")
plt.ylabel("Number of Emails")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="target",
    y="word_count"
)

plt.title("Email Word Count by Class")
plt.xlabel("Class (0 = Ham, 1 = Spam)")
plt.ylabel("Word Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="target",
    y="character_count"
)

plt.title("Email Character Count by Class")
plt.xlabel("Class (0 = Ham, 1 = Spam)")
plt.ylabel("Character Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="target",
    y="url_count"
)

plt.title("Number of URLs by Email Class")
plt.xlabel("Class (0 = Ham, 1 = Spam)")
plt.ylabel("URL Count")

plt.tight_layout()
plt.show()

In [ ]:
# 16. Experiment 1 — Classification Before Outlier Removal
# This is the baseline experiment.
# Important:** outliers have NOT been removed at this stage.
# The model uses TF-IDF text features and Logistic Regression.


In [ ]:
df_before_outliers = df.copy()

print(
    "Number of emails before outlier removal:",
    len(df_before_outliers)
)

In [ ]:
X = df_before_outliers["clean_text"]
y = df_before_outliers["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# 17. TF-IDF Vectorization Before Outlier Removal
# TF-IDF converts the email text into numerical vectors.
# The vectorizer is fitted only on the training data to avoid data leakage.


In [ ]:
tfidf_before = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf_before.fit_transform(X_train)
X_test_tfidf = tfidf_before.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

In [ ]:
# 18. Logistic Regression Before Outlier Removal


In [ ]:
model_before = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

model_before.fit(
    X_train_tfidf,
    y_train
)

y_pred_before = model_before.predict(
    X_test_tfidf
)

In [ ]:
# 19. Evaluate the Baseline Model
# Accuracy alone is not sufficient, so accuracy, precision, recall, and F1-score are reported.


In [ ]:
accuracy_before = accuracy_score(
    y_test,
    y_pred_before
)

precision_before = precision_score(
    y_test,
    y_pred_before,
    zero_division=0
)

recall_before = recall_score(
    y_test,
    y_pred_before,
    zero_division=0
)

f1_before = f1_score(
    y_test,
    y_pred_before,
    zero_division=0
)

print("MODEL PERFORMANCE BEFORE OUTLIER REMOVAL")
print("-----------------------------------------")
print(f"Accuracy : {accuracy_before:.4f}")
print(f"Precision: {precision_before:.4f}")
print(f"Recall   : {recall_before:.4f}")
print(f"F1 Score : {f1_before:.4f}")

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_before,
        target_names=["Ham", "Spam"],
        zero_division=0
    )
)

In [ ]:
# 20. Confusion Matrix Before Outlier Removal


In [ ]:
cm_before = confusion_matrix(
    y_test,
    y_pred_before
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_before,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)

plt.title("Confusion Matrix - Before Outlier Removal")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# 21. Outlier Detection Using the IQR Method
# The Interquartile Range method identifies observations outside:
# Lower bound = Q1 - 1.5 × IQR
# Upper bound = Q3 + 1.5 × IQR
# The outlier analysis is performed on email-level numerical features rather than directly on raw text.


In [ ]:
outlier_features = [
    "character_count",
    "word_count",
    "sentence_count",
    "url_count",
    "digit_count",
    "uppercase_count",
    "special_character_count"
]

In [ ]:
outlier_information = []

for column in outlier_features:

    Q1 = df_before_outliers[column].quantile(0.25)
    Q3 = df_before_outliers[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_information.append([
        column,
        Q1,
        Q3,
        IQR,
        lower_bound,
        upper_bound
    ])

outlier_table = pd.DataFrame(
    outlier_information,
    columns=[
        "Feature",
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound"
    ]
)

display(outlier_table)

In [ ]:
# 22. Detect Outlier Rows


In [ ]:
outlier_mask = pd.Series(
    False,
    index=df_before_outliers.index
)

for column in outlier_features:

    Q1 = df_before_outliers[column].quantile(0.25)
    Q3 = df_before_outliers[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    current_outliers = (
        (df_before_outliers[column] < lower_bound) |
        (df_before_outliers[column] > upper_bound)
    )

    outlier_mask = outlier_mask | current_outliers

number_of_outliers = int(outlier_mask.sum())

percentage_outliers = (
    number_of_outliers /
    len(df_before_outliers)
) * 100

print("Number of detected outliers:", number_of_outliers)
print(
    "Percentage of dataset detected as outliers:",
    f"{percentage_outliers:.2f}%"
)

In [ ]:
# 23. Outlier Visualizations
# Boxplots are used to visually identify extreme values in the numerical email features.


In [ ]:
for column in outlier_features:

    plt.figure(figsize=(9, 4))

    sns.boxplot(
        x=df_before_outliers[column]
    )

    plt.title(
        f"Boxplot Before Outlier Removal: {column}"
    )

    plt.xlabel(column)

    plt.tight_layout()
    plt.show()

In [ ]:
# 24. Experiment 2 — Remove Outliers
# The outlier rows are now removed to create the second version of the dataset.
# This step happens only after the baseline experiment has been completed.


In [ ]:
df_after_outliers = df_before_outliers[
    ~outlier_mask
].copy()

print(
    "Dataset before outlier removal:",
    df_before_outliers.shape
)

print(
    "Dataset after outlier removal:",
    df_after_outliers.shape
)

print(
    "Rows removed:",
    len(df_before_outliers) - len(df_after_outliers)
)

In [ ]:
# 25. Check Class Distribution After Outlier Removal


In [ ]:
print("Class distribution BEFORE outlier removal:")
print(df_before_outliers["target"].value_counts())

print("\nClass distribution AFTER outlier removal:")
print(df_after_outliers["target"].value_counts())

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

sns.countplot(
    data=df_before_outliers,
    x="target",
    ax=axes[0]
)

axes[0].set_title("Before Outlier Removal")
axes[0].set_xlabel("Class (0 = Ham, 1 = Spam)")
axes[0].set_ylabel("Number of Emails")

sns.countplot(
    data=df_after_outliers,
    x="target",
    ax=axes[1]
)

axes[1].set_title("After Outlier Removal")
axes[1].set_xlabel("Class (0 = Ham, 1 = Spam)")
axes[1].set_ylabel("Number of Emails")

plt.tight_layout()
plt.show()

In [ ]:
# 26. Visualize the Numerical Features After Outlier Removal


In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df_after_outliers,
    x="target",
    y="word_count"
)

plt.title("Word Count by Class After Outlier Removal")
plt.xlabel("Class (0 = Ham, 1 = Spam)")
plt.ylabel("Word Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df_after_outliers,
    x="target",
    y="character_count"
)

plt.title("Character Count by Class After Outlier Removal")
plt.xlabel("Class (0 = Ham, 1 = Spam)")
plt.ylabel("Character Count")

plt.tight_layout()
plt.show()

In [ ]:
# 27. Classification After Outlier Removal
# The same Logistic Regression model and TF-IDF approach are now applied to the dataset after outlier removal.


In [ ]:
X_after = df_after_outliers["clean_text"]
y_after = df_after_outliers["target"]

X_train_after, X_test_after, y_train_after, y_test_after = train_test_split(
    X_after,
    y_after,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_after
)

print("Training samples after removal:", len(X_train_after))
print("Testing samples after removal:", len(X_test_after))

In [ ]:
# 28. TF-IDF After Outlier Removal


In [ ]:
tfidf_after = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf_after = tfidf_after.fit_transform(
    X_train_after
)

X_test_tfidf_after = tfidf_after.transform(
    X_test_after
)

In [ ]:
# 29. Logistic Regression After Outlier Removal


In [ ]:
model_after = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

model_after.fit(
    X_train_tfidf_after,
    y_train_after
)

y_pred_after = model_after.predict(
    X_test_tfidf_after
)

In [ ]:
# 30. Evaluate the Model After Outlier Removal


In [ ]:
accuracy_after = accuracy_score(
    y_test_after,
    y_pred_after
)

precision_after = precision_score(
    y_test_after,
    y_pred_after,
    zero_division=0
)

recall_after = recall_score(
    y_test_after,
    y_pred_after,
    zero_division=0
)

f1_after = f1_score(
    y_test_after,
    y_pred_after,
    zero_division=0
)

print("MODEL PERFORMANCE AFTER OUTLIER REMOVAL")
print("----------------------------------------")
print(f"Accuracy : {accuracy_after:.4f}")
print(f"Precision: {precision_after:.4f}")
print(f"Recall   : {recall_after:.4f}")
print(f"F1 Score : {f1_after:.4f}")

In [ ]:
print(
    classification_report(
        y_test_after,
        y_pred_after,
        target_names=["Ham", "Spam"],
        zero_division=0
    )
)

In [ ]:
# 31. Confusion Matrix After Outlier Removal


In [ ]:
cm_after = confusion_matrix(
    y_test_after,
    y_pred_after
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_after,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)

plt.title("Confusion Matrix - After Outlier Removal")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# 32. Final Before vs After Comparison


In [ ]:
comparison = pd.DataFrame(
    {
        "Before Outlier Removal": [
            accuracy_before,
            precision_before,
            recall_before,
            f1_before
        ],
        "After Outlier Removal": [
            accuracy_after,
            precision_after,
            recall_after,
            f1_after
        ]
    },
    index=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

display(comparison.round(4))

In [ ]:
# 33. Performance Comparison Visualization


In [ ]:
comparison.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Spam Classification Performance Before vs After Outlier Removal"
)

plt.xlabel("Evaluation Metric")
plt.ylabel("Score")

plt.ylim(0, 1.05)

plt.xticks(rotation=0)

plt.legend(title="Experiment")

plt.tight_layout()
plt.show()

In [ ]:
# 34. Measure the Change in Performance


In [ ]:
improvement = (
    comparison["After Outlier Removal"]
    - comparison["Before Outlier Removal"]
)

print("Change in performance after outlier removal:")
print()

for metric, change in improvement.items():

    if change > 0:
        print(
            f"{metric}: improved by {change:.4f}"
        )

    elif change < 0:
        print(
            f"{metric}: decreased by {abs(change):.4f}"
        )

    else:
        print(
            f"{metric}: no change"
        )

In [ ]:
# 35. Final Dataset Summary


In [ ]:
original_rows = len(df)

rows_before_outlier_removal = len(
    df_before_outliers
)

rows_after_outlier_removal = len(
    df_after_outliers
)

rows_removed = (
    rows_before_outlier_removal
    - rows_after_outlier_removal
)

percentage_removed = (
    rows_removed /
    rows_before_outlier_removal
) * 100

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(
    f"Rows after basic preprocessing: {original_rows}"
)

print(
    f"Rows before outlier removal: {rows_before_outlier_removal}"
)

print(
    f"Outliers removed: {rows_removed}"
)

print(
    f"Rows after outlier removal: {rows_after_outlier_removal}"
)

print(
    f"Percentage removed: {percentage_removed:.2f}%"
)

print("\nModel Performance:")
display(comparison.round(4))

In [ ]:
# 36. Conclusion
# The final comparison should be interpreted using the actual results produced above.
# Important points to discuss:
# Whether outlier removal increased or decreased accuracy.
# Whether precision improved.
